In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [11]:

!pip install sentence-transformers faiss-cpu -q

In [3]:
df = pd.read_csv("/kaggle/input/datasets/ravirajbabasomane/amazon-reviews-2023/Amazon_reviews_2023.csv")
df['text']  = df['text'].fillna('')
df['title'] = df['title'].fillna('')
print(f"Loaded: {df.shape}")

Loaded: (701528, 10)


In [15]:
# Better item representation using review text signals
item_descriptions = df.groupby('parent_asin').agg(
    # Most helpful review title as proxy for product name
    title        = ('title', lambda x: max(x, key=len)),  # longest title = most descriptive
    reviews      = ('text', lambda x: ' '.join(
                        [str(i) for i in list(x)[:5] if pd.notna(i)]
                   )),
    avg_rating   = ('rating', 'mean'),
    review_count = ('rating', 'count')
).reset_index()

# Build rich embed text — this is what gets searched
item_descriptions['embed_text'] = (
    "Product: "  + item_descriptions['title'].fillna('') + ". " +
    "Customer reviews say: " + item_descriptions['reviews'].str[:400]
)

print(f"Items to embed: {len(item_descriptions):,}")
print("\nSample embed text:")
print(item_descriptions['embed_text'].iloc[10])

Items to embed: 112,565

Sample embed text:
Product: A jaded view from the top. Customer reviews say: His points are as obvious as his title. A successful career and life from a singular point of view. Mr Papone oversaw some of this centuries most successful advertising and his analysis of that process is interesting. Having spent time as an employee of Ogilvy Mather I cannot share the same level of love he has for that firm and its work but that is another story. If you want to read yet another b


In [5]:
# Cell 4 — Embed items (this will take ~10-15 mins on Kaggle T4)


embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Embed in batches to avoid memory issues
texts = item_descriptions['embed_text'].tolist()

print("Embedding items... (grab a drink, this takes a few minutes)")
embeddings = embedder.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding items... (grab a drink, this takes a few minutes)


Batches:   0%|          | 0/440 [00:00<?, ?it/s]

Embeddings shape: (112565, 384)


In [12]:
import faiss
import numpy as np
import pickle

# Convert embeddings to float32 (FAISS requirement)
embeddings_matrix = np.array(embeddings).astype('float32')

# Normalize for cosine similarity
faiss.normalize_L2(embeddings_matrix)

# Build the index
dimension = embeddings_matrix.shape[1]  # 384 for MiniLM
index = faiss.IndexFlatIP(dimension)    # Inner Product = cosine after normalization
index.add(embeddings_matrix)

print(f"FAISS index ready. Total items: {index.ntotal}")

# Save index and metadata so you don't have to rebuild
faiss.write_index(index, "items.index")

# Save the item metadata alongside
item_meta = item_descriptions[['parent_asin', 'title', 'avg_rating', 'review_count']].reset_index(drop=True)
item_meta.to_pickle("item_meta.pkl")

print("Saved: items.index + item_meta.pkl")

FAISS index ready. Total items: 112565
Saved: items.index + item_meta.pkl


In [18]:
def retrieve_items(query_text, n_results=10, min_reviews=3):
    """
    Given a query string, retrieve most relevant items using FAISS.
    """
    # Embed the query
    query_vec = embedder.encode([query_text]).astype('float32')
    faiss.normalize_L2(query_vec)

    # Search
    scores, indices = index.search(query_vec, n_results * 2)

    items = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = item_meta.iloc[idx]
        if meta['review_count'] >= min_reviews:
            items.append({
                'asin'         : meta['parent_asin'],
                'title'        : meta['title'],
                'avg_rating'   : meta['avg_rating'],
                'review_count' : meta['review_count'],
                'similarity'   : float(score)
            })

    return sorted(items, key=lambda x: x['similarity'], reverse=True)[:n_results]


# Test it
test_results = retrieve_items("moisturising face cream for dry skin", n_results=5)
print("RETRIEVAL TEST RESULTS:")
for i, item in enumerate(test_results, 1):
    print(f"{i}. {item['title'][:60]}")
    print(f"   ⭐ {item['avg_rating']:.1f} | {item['review_count']} reviews | sim: {item['similarity']:.3f}")

RETRIEVAL TEST RESULTS:
1. I highly recommend this cream for dry skin!!!
   ⭐ 3.8 | 5 reviews | sim: 0.781
2. DayTime Moisturizer for Dry Skin
   ⭐ 3.8 | 8 reviews | sim: 0.733
3. Okay, but not for super dry skin
   ⭐ 3.7 | 13 reviews | sim: 0.720


In [19]:
def build_retrieval_query(answers):
    """
    Convert elicitation answers into a retrieval-friendly query.
    Positively framed — negations confuse embedding search.
    """
    product_type = answers['q1']  # e.g. skincare
    priority     = answers['q2']  # e.g. price
    avoid        = answers['q3']  # e.g. alcohol

    # Positive framing only — no negations in the query
    query = (
        f"gentle {product_type} product. "
        f"affordable and good value. "
        f"natural ingredients. fragrance-free. gentle formula."
    )

    return query, avoid  # return avoid separately for post-filtering


def post_filter(items, avoid_keyword):
    """
    After retrieval, remove items whose reviews mention the avoided ingredient.
    """
    avoid = avoid_keyword.lower()
    filtered = []

    for item in items:
        # Check if the item's reviews mention the avoided thing positively
        item_text = item['title'].lower()
        # Simple heuristic: if title mentions it as a negative → skip
        if avoid in item_text:
            continue
        filtered.append(item)

    return filtered


def run_elicitation():
    print("Let me help you find products you'll love.")
    print("Just answer 3 quick questions:\n")

    answers = {}
    questions = [
        ("q1", "What type of beauty/personal care products do you use most often?",
               "(e.g. skincare, haircare, fragrance, makeup)"),
        ("q2", "What matters most to you when buying a product?",
               "(e.g. price, natural ingredients, brand reputation, effectiveness)"),
        ("q3", "Any ingredients or product types you avoid?",
               "(e.g. alcohol, strong fragrances, sulphates, oily textures)")
    ]

    for qid, question, example in questions:
        print(f"Q: {question}")
        print(f"   {example}")
        answers[qid] = input("Your answer: ").strip()
        print()

    query, avoid = build_retrieval_query(answers)
    print(f"Search query : {query}")
    print(f"Filtering out: {avoid}\n")

    # Retrieve more than needed so filtering doesn't empty the list
    raw_results = retrieve_items(query, n_results=20)

    # Post-filter
    filtered = post_filter(raw_results, avoid)

    # Take top 5
    final = filtered[:5]

    print("RECOMMENDATIONS:")
    print("="*60)
    for i, item in enumerate(final, 1):
        print(f"{i}. {item['title'][:70]}")
        print(f"   ⭐ {item['avg_rating']} | {item['review_count']} reviews")
        print()

    return final

# Run it
results = run_elicitation()

Let me help you find products you'll love.
Just answer 3 quick questions:

Q: What type of beauty/personal care products do you use most often?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  skincare



Q: What matters most to you when buying a product?
   (e.g. price, natural ingredients, brand reputation, effectiveness)


Your answer:  price



Q: Any ingredients or product types you avoid?
   (e.g. alcohol, strong fragrances, sulphates, oily textures)


Your answer:  alcohol



Search query : gentle skincare product. affordable and good value. natural ingredients. fragrance-free. gentle formula.
Filtering out: alcohol

RECOMMENDATIONS:
1. Gentle and soothing
   ⭐ 5.0 | 3 reviews

2. Gentle
   ⭐ 5.0 | 3 reviews

3. Gentle on your skin
   ⭐ 5.0 | 3 reviews

4. Your skin will love it!
   ⭐ 4.818181818181818 | 11 reviews

5. Soothing cleanse results in silky skin
   ⭐ 5.0 | 3 reviews



In [20]:
import math

def confidence_score(avg_rating, review_count, prior_rating=3.96, prior_count=10):
    """
    Bayesian average — balances rating with review count.
    A 4.5 rating with 50 reviews beats a 5.0 rating with 3 reviews.
    prior_rating = dataset mean (3.96 from your EDA)
    prior_count  = minimum reviews before we trust the rating
    """
    return (
        (prior_count * prior_rating + review_count * avg_rating) /
        (prior_count + review_count)
    )

def retrieve_items(query_text, n_results=10, min_reviews=3):
    query_vec = embedder.encode([query_text]).astype('float32')
    faiss.normalize_L2(query_vec)

    scores, indices = index.search(query_vec, n_results * 3)

    items = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = item_meta.iloc[idx]
        if meta['review_count'] < min_reviews:
            continue

        conf = confidence_score(meta['avg_rating'], meta['review_count'])

        items.append({
            'asin'          : meta['parent_asin'],
            'title'         : meta['title'],
            'avg_rating'    : round(meta['avg_rating'], 2),
            'review_count'  : int(meta['review_count']),
            'similarity'    : float(score),
            'confidence'    : round(conf, 3),
            # Final score = blend of semantic similarity + rating confidence
            'final_score'   : float(score) * 0.7 + (conf / 5.0) * 0.3
        })

    # Sort by final blended score
    return sorted(items, key=lambda x: x['final_score'], reverse=True)[:n_results]

In [21]:
def run_elicitation():
    print("Let me help you find products you'll love.")
    print("Just answer 3 quick questions:\n")

    answers = {}
    questions = [
        ("q1", "What type of beauty/personal care products do you use most often?",
               "(e.g. skincare, haircare, fragrance, makeup)"),
        ("q2", "What matters most to you when buying a product?",
               "(e.g. price, natural ingredients, brand reputation, effectiveness)"),
        ("q3", "Any ingredients or product types you avoid?",
               "(e.g. alcohol, strong fragrances, sulphates, oily textures)")
    ]

    for qid, question, example in questions:
        print(f"Q: {question}")
        print(f"   {example}")
        answers[qid] = input("Your answer: ").strip()
        print()

    query, avoid = build_retrieval_query(answers)
    print(f"Search query : {query}")
    print(f"Filtering out: {avoid}\n")

    raw_results  = retrieve_items(query, n_results=20)
    filtered     = post_filter(raw_results, avoid)
    final        = filtered[:5]

    print("RECOMMENDATIONS:")
    print("="*60)
    for i, item in enumerate(final, 1):
        print(f"{i}. {item['title'][:70]}")
        print(f"   ASIN       : {item['asin']}")
        print(f"   ⭐ Rating  : {item['avg_rating']} ({item['review_count']} reviews)")
        print(f"   Confidence : {item['confidence']}")
        print(f"   Final score: {item['final_score']:.4f}")
        print()

    return final, answers

results, answers = run_elicitation()

Let me help you find products you'll love.
Just answer 3 quick questions:

Q: What type of beauty/personal care products do you use most often?
   (e.g. skincare, haircare, fragrance, makeup)


Your answer:  skincare



Q: What matters most to you when buying a product?
   (e.g. price, natural ingredients, brand reputation, effectiveness)


Your answer:  price



Q: Any ingredients or product types you avoid?
   (e.g. alcohol, strong fragrances, sulphates, oily textures)


Your answer:  alcohol



Search query : gentle skincare product. affordable and good value. natural ingredients. fragrance-free. gentle formula.
Filtering out: alcohol

RECOMMENDATIONS:
1. Gentle and soothing
   ASIN       : B07CX6VHTP
   ⭐ Rating  : 5.0 (3 reviews)
   Confidence : 4.2
   Final score: 0.7734

2. Your skin will love it!
   ASIN       : B00WTCKGBK
   ⭐ Rating  : 4.82 (11 reviews)
   Confidence : 4.41
   Final score: 0.7733

3. Gentle
   ASIN       : B09NXH42C4
   ⭐ Rating  : 5.0 (3 reviews)
   Confidence : 4.2
   Final score: 0.7694

4. Gentle on your skin
   ASIN       : B000FDB9N4
   ⭐ Rating  : 5.0 (3 reviews)
   Confidence : 4.2
   Final score: 0.7663

5. and found that it is gentle and skin feels great. first bought it at t
   ASIN       : B01IAQGN6K
   ⭐ Rating  : 5.0 (5 reviews)
   Confidence : 4.307
   Final score: 0.7598

